In [ ]:
from scapyter.ui.trace_plotter import TracePlotter
from scapyter.infrastructure.h5_project_file_reader import H5ProjectFileReader
from scapyter.domain.analysis.correlation.cpa import CpaCorrelation
from scapyter.domain.leakage.leakage import SboxOutputLeakageModel
from scapyter.domain.value_object import RangeParameters, Range, DataSource
from scapyter.ui.progressive_cpa_plotter import ProgressiveCpaPlotter
from scapyter.application.analysis.correlation.progressive_correlation_service import ProgressiveCorrelationService
from scapyter.domain.analysis.correlation.value_objects.correlation_functions import CorrelationFunction


file_path="../../data/smart-card-project/smart_card_project.sx"

trace_repo = H5ProjectFileReader(file_path)
trace_plotter = TracePlotter(trace_repo)
trace_plotter.plot_single(index=0)

range_parameter = RangeParameters(
    trace_range=Range(0, 125),
    sample_range=Range(0 , 44613)
)

correlation_functions = [
    CorrelationFunction(
        byte_location=i,
        leakage_model=SboxOutputLeakageModel(),
        correlation=CpaCorrelation(),
    )
    for i in range(2)
]

service = ProgressiveCorrelationService(
    range_parameters=range_parameter,
    project_file_reader=trace_repo,
    data_source=DataSource.PLAINTEXT,
    correlation_functions=correlation_functions,
)

results = service.run(progress_steps=50)

plotter = ProgressiveCpaPlotter(list(results))

plotter.plot_convergence(
    byte_index=0,
    correct_key=0x00,
)

plotter.plot_convergence(
    byte_index=1,
    correct_key=0x01,
)